Building tiny gpt and training it on The Project Gutenberg eBook of The Age of Fable

In [1]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  839083


In [3]:
# let's look at the first 1000 characters
print(text[:1000])

BULFINCH'S MYTHOLOGY

THE AGE OF FABLE

THE AGE OF CHIVALRY

LEGENDS OF CHARLEMAGNE

BY THOMAS BULFINCH

COMPLETE IN ONE VOLUME


[Editor's Note: The etext contains only THE AGE OF FABLE]




PUBLISHERS' PREFACE


No new edition of Bulfinch's classic work can be considered
complete without some notice of the American scholar to whose wide
erudition and painstaking care it stands as a perpetual monument.
"The Age of Fable" has come to be ranked with older books like
"Pilgrim's Progress," "Gulliver's Travels," "The Arabian Nights,"
"Robinson Crusoe," and five or six other productions of world-wide
renown as a work with which every one must claim some acquaintance
before his education can be called really complete. Many readers
of the present edition will probably recall coming in contact with
the work as children, and, it may be added, will no doubt discover
from a fresh perusal the source of numerous bits of knowledge that
have remained stored in their minds since those early years. Yet

In [4]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"#$%'()*+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz—‘’“”•™
90


In [6]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("I like shawarma"))
print(decode(encode("I like shawarma")))

[37, 1, 68, 65, 67, 61, 1, 75, 64, 57, 79, 57, 74, 69, 57]
I like shawarma


In [ ]:
#Encoding the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([839083]) torch.int64
tensor([30, 49, 40, 34, 37, 42, 31, 36,  7, 47,  1, 41, 53, 48, 36, 43, 40, 43,
        35, 53,  0,  0, 48, 36, 33,  1, 29, 35, 33,  1, 43, 34,  1, 34, 29, 30,
        40, 33,  0,  0, 48, 36, 33,  1, 29, 35, 33,  1, 43, 34,  1, 31, 36, 37,
        50, 29, 40, 46, 53,  0,  0, 40, 33, 35, 33, 42, 32, 47,  1, 43, 34,  1,
        31, 36, 29, 46, 40, 33, 41, 29, 35, 42, 33,  0,  0, 30, 53,  1, 48, 36,
        43, 41, 29, 47,  1, 30, 49, 40, 34, 37, 42, 31, 36,  0,  0, 31, 43, 41,
        44, 40, 33, 48, 33,  1, 37, 42,  1, 43, 42, 33,  1, 50, 43, 40, 49, 41,
        33,  0,  0,  0, 55, 33, 60, 65, 76, 71, 74,  7, 75,  1, 42, 71, 76, 61,
        26,  1, 48, 64, 61,  1, 61, 76, 61, 80, 76,  1, 59, 71, 70, 76, 57, 65,
        70, 75,  1, 71, 70, 68, 81,  1, 48, 36, 33,  1, 29, 35, 33,  1, 43, 34,
         1, 34, 29, 30, 40, 33, 56,  0,  0,  0,  0,  0, 44, 49, 30, 40, 37, 47,
        36, 33, 46, 47,  7,  1, 44, 46, 33, 34, 29, 31, 33,  0,  0,  0, 42, 71,
       

In [8]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

tensor([30, 49, 40, 34, 37, 42, 31, 36,  7])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([30]) the target: 49
when input is tensor([30, 49]) the target: 40
when input is tensor([30, 49, 40]) the target: 34
when input is tensor([30, 49, 40, 34]) the target: 37
when input is tensor([30, 49, 40, 34, 37]) the target: 42
when input is tensor([30, 49, 40, 34, 37, 42]) the target: 31
when input is tensor([30, 49, 40, 34, 37, 42, 31]) the target: 36
when input is tensor([30, 49, 40, 34, 37, 42, 31, 36]) the target: 7


In [11]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 1, 64, 57, 78, 61,  1, 68, 71],
        [65, 75, 61,  0,  1,  1,  1,  1],
        [71, 78, 61, 74,  1, 64, 61, 74],
        [65, 63, 70, 75,  1, 57, 70, 60]])
targets:
torch.Size([4, 8])
tensor([[64, 57, 78, 61,  1, 68, 71, 75],
        [75, 61,  0,  1,  1,  1,  1, 29],
        [78, 61, 74,  1, 64, 61, 74,  0],
        [63, 70, 75,  1, 57, 70, 60,  1]])
----
when input is [1] the target: 64
when input is [1, 64] the target: 57
when input is [1, 64, 57] the target: 78
when input is [1, 64, 57, 78] the target: 61
when input is [1, 64, 57, 78, 61] the target: 1
when input is [1, 64, 57, 78, 61, 1] the target: 68
when input is [1, 64, 57, 78, 61, 1, 68] the target: 71
when input is [1, 64, 57, 78, 61, 1, 68, 71] the target: 75
when input is [65] the target: 75
when input is [65, 75] the target: 61
when input is [65, 75, 61] the target: 0
when input is [65, 75, 61, 0] the target: 1
when input is [65, 75, 61, 0, 1] the target: 1
when input is [65, 75, 61,

In [12]:
print(xb) # our input to the transformer

tensor([[ 1, 64, 57, 78, 61,  1, 68, 71],
        [65, 75, 61,  0,  1,  1,  1,  1],
        [71, 78, 61, 74,  1, 64, 61, 74],
        [65, 63, 70, 75,  1, 57, 70, 60]])
